# Kajiado North — Load, Clean, and Initial Analysis
This notebook loads the voter CSV, cleans key fields, filters to the Kajiado North constituency, performs basic QA, and produces summary tables and simple plots.

In [ ]:
# Cell 2: Imports and display settings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
sns.set(style='whitegrid')
pd.options.display.max_columns = 200
pd.options.display.width = 120

In [ ]:
# Cell 3: Load raw CSV
DATA_PATH = '034 - 034.csv (1).csv'
# load everything as string first to avoid parsing problems
df_raw = pd.read_csv(DATA_PATH, dtype=str, encoding='utf-8', keep_default_na=False)
print('Rows loaded:', len(df_raw))
print('Columns:', list(df_raw.columns))
df_raw.head()

In [ ]:
# Cell 4: Cleaning helpers
def clean_text_fields(df):
    text_cols = ['county','constituency','caw','polling_center','polling_station','pstream','sex']
    for col in text_cols:
        if col in df.columns:
            df[col] = df[col].astype(str).str.strip().replace({'NULL':'', 'None':'', 'nan':''}).str.upper()
    return df

def derive_full_name(df):
    for col in ['fname','mname','sname']:
        if col not in df.columns:
            df[col] = ''
    df['full_name'] = (df['fname'].fillna('') + ' ' + df['mname'].fillna('') + ' ' + df['sname'].fillna('')).str.replace('\s+', ' ', regex=True).str.strip()
    return df

def parse_birth_dates(df, reference_year=2026):
    if 'date_of_birth' in df.columns:
        df['birth_date'] = pd.to_datetime(df['date_of_birth'], dayfirst=True, errors='coerce')
        df['birth_year'] = df['birth_date'].dt.year
        df['age'] = np.where(df['birth_year'].notnull(), reference_year - df['birth_year'], np.nan)
    else:
        df['birth_date'] = pd.NaT
        df['birth_year'] = np.nan
        df['age'] = np.nan
    return df


def count_missing(df):
    return df.isin(['', ' ', None, np.nan]).sum()

In [ ]:
# **Apply cleaning and derive columns**
df = df_raw.copy()
df = clean_text_fields(df)
df = derive_full_name(df)
df = parse_birth_dates(df)

# Standardize id column name if different
if 'id_passport_no' not in df.columns:
    if 'id' in df.columns:
        df = df.rename(columns={'id':'id_passport_no'})

print('After cleaning — sample:')
df[['constituency','caw','polling_center','polling_station','sex','birth_date','age','full_name']].head()

In [ ]:
# Filter to Kajiado North and quick QA

const_name = 'KAJIADO NORTH'
mask = df['constituency'].fillna('').str.upper() == const_name
df_kn = df[mask].copy()
print('Total records in Kajiado North:', len(df_kn))

# Missing value counts (top fields)
print('\nMissing counts (Kajiado North subset):')
print(count_missing(df_kn)[['date_of_birth','sex','id_passport_no']].to_string())

# Duplicates by id
if 'id_passport_no' in df_kn.columns:
    dup_ids = df_kn['id_passport_no'].duplicated(keep=False)
    print('\nDuplicate id_passport_no count:', dup_ids.sum())

# Age distribution quick stats
print('\nAge stats (Kajiado North):')
print(df_kn['age'].describe())

# Age bands
bins = [0,17,24,34,44,54,64,200]
labels = ['<18','18-24','25-34','35-44','45-54','55-64','65+']
df_kn['age_band'] = pd.cut(df_kn['age'], bins=bins, labels=labels, include_lowest=True)
print('\nAge band counts:')
print(df_kn['age_band'].value_counts().sort_index())

In [ ]:
# Summaries and simple plots
# Sex distribution
sex_counts = df_kn['sex'].value_counts(dropna=False)
print('Sex distribution:\n', sex_counts)

# Top CAWs
if 'caw' in df_kn.columns:
    top_caws = df_kn['caw'].value_counts().head(10)
    print('\nTop 10 CAWs:\n', top_caws)

# Plots (safe: only show if non-empty)
plt.figure(figsize=(8,4))
if not sex_counts.empty:
    sex_counts.plot(kind='bar', title='Kajiado North — Sex distribution')
    plt.ylabel('Voter count')
    plt.show()

plt.figure(figsize=(8,4))
if 'age' in df_kn.columns and df_kn['age'].notnull().sum()>0:
    sns.histplot(df_kn['age'].dropna(), bins=20)
    plt.title('Kajiado North — Age distribution')
    plt.xlabel('Age')
    plt.show()

if 'caw' in df_kn.columns and not top_caws.empty:
    plt.figure(figsize=(10,5))
    top_caws.plot(kind='bar')
    plt.title('Top 10 CAWs by registered voters')
    plt.ylabel('Voter count')
    plt.show()